In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import pandas as pd
from IPython.display import display, Markdown
import pycountry_convert as pc
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
from matplotlib.colors import Normalize
from matplotlib.pyplot import get_cmap
from matplotlib.cm import ScalarMappable
import seaborn as sns

from emu_renewal.constants import MOB_SOURCE_COLOURS, OUTPUTS_PATH, ANALYSIS_NAMES
from emu_renewal.utils import get_country_short_name
from emu_renewal.outputs import get_idatas_for_mob_type
from emu_renewal.outputs import get_param_mean_by_country
from emu_renewal.plotting import get_detailed_param_results
from emu_renewal.outputs import get_param_vals_by_analysis, get_ratios_from_disps, get_median_ratios
from emu_renewal.plotting import get_avail_groupings

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "46693102"
all_countries = ls(job_path)
# countries_by_cont = get_countries_by_continent(countries)
param = "mob_exp"

# Purpose
This document presents estimates of 
the mobility exponent parameter posteriors by country and mobility analysis type.
This is intended to provide an estimate of the strength of the mobility effects.
Values from zero to two were considered plausible 
(the prior for this parameter was uniform over domain $[0, 2]$). 
A value of zero represents no effect, 
a value of one represents a linear scaling in transmission with mobility changes,
and a value of two represents a quadratic scaling in transmission with mobility changes 
(which could be epidemiologically considered as an effect on both infector and infectee).

# Mobility exponent parameter posteriors by country

In [ ]:
mob_source = "g_mob"

In [ ]:
# Get the dispersion ratios
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in all_countries}
ratio_dists = get_ratios_from_disps(disp_posts)
ratio_vals = get_median_ratios(ratio_dists, mob_source)
ratios = {get_country_short_name(k): v for k, v in ratio_vals.items()}

In [ ]:
grouping = get_avail_groupings(ratio_vals.keys())

In [ ]:
idatas, _ = get_idatas_for_mob_type(job_path, all_countries, mob_source)
mob_exp_df = pd.DataFrame(columns=idatas)
for iso3 in idatas:
    mob_exp_df[iso3] = idatas[iso3].posterior["mob_exp"].to_dataframe()

In [ ]:
fig, axes = subplots(3, 5, figsize=[12, 12], sharey=True)
flat_axes = axes.ravel()

norm = Normalize(vmin=min(ratios.values()), vmax=max(ratios.values()))
cmap = get_cmap(f"{MOB_SOURCE_COLOURS[mob_source].capitalize()}s_r")
palette = {c: cmap(norm(v)) for c, v in ratios.items()}

for r, (region, countries) in enumerate(grouping.items()):
    ax = flat_axes[r]
    ax.set_title(region)
    plot_df = mob_exp_df[countries]
    plot_df.columns = plot_df.columns.map(get_country_short_name)
    sns.violinplot(plot_df, ax=ax, palette=palette)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=90)

ax = flat_axes[r + 1]
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, location="left")
ax.remove()

fig.tight_layout()